In [15]:
import pandas as pd
from itertools import combinations
from IPython.display import display

## Group Metadata Notebook:
#### We want to create recording groups to be able to do stats on our spike parameterization to figure out what metadata parameters result in spike waveform differences. We need to create groups for this comparison because because each recording has multiple variables (species, age, sex, brainorigin, somalayer, dendritic type), that varies for each recording. Thus, when we ran the stats for spike params prior to the group separation, we didn't know which metadata params were generating the differences/influencing the changes in spike waveform.

#### I initially was doing the grouping manually, but now doing automized version (below)

#### Note: I am ignoring age (all are adults?)and weight metadata params (for now)

#### Helper functions 

In [22]:
def load_dataframe_from_pickle(file_path):
    """
    Function to extract a DataFrame from a pickle file.
    
    Args:
    - file_path (str): Path to the pickle file.
    
    Returns:
    - dataframe (pd.DataFrame): Loaded DataFrame.
    """
    dataframe = pd.read_pickle(file_path)
    return dataframe

### Read in pickle with filtered parameterized data
#### This pandas dataframe contains spike param data and animal/cell metadata for each spike(one spike per row). The dataframe has already been filtered for rsq fits (see other notebooks for rsq thresholds)

In [23]:
allMonkey_df = load_dataframe_from_pickle(r"/Users/blancamartin/Downloads/allMonkey_df_filt.pkl")

In [24]:
allMonkey_df

,ramp_amp,inflection_time,inflection_amp,peak_amp,peak_width,peak_sharpness,exp_lambda,exp_const,isi,r_squared_ramp,...,Sweep_#,file_name,dendriticType,SomaLayerLoc,brainOrigin,Weight,Sex,Age,Species,Monkey ID
4,1.832248,0.60,-38.827516,11.984253,0.55,3.929138,5.981717,-53.646425,26.60,0.950116,...,Sweep_10,M03_JS_A1_C01,A,3,PFC,6.3,F,9.96,Macaca fascicularis,M03
5,2.422453,0.60,-37.515260,11.083985,0.60,3.491211,6.095568,-52.587629,29.75,0.994699,...,Sweep_10,M03_JS_A1_C01,A,3,PFC,6.3,F,9.96,Macaca fascicularis,M03
6,1.725643,0.60,-38.571168,11.453247,0.55,3.533936,6.194347,-53.275150,28.45,0.969743,...,Sweep_10,M03_JS_A1_C01,A,3,PFC,6.3,F,9.96,Macaca fascicularis,M03
7,2.304789,0.60,-38.330079,11.434937,0.55,3.767395,5.654485,-53.285600,27.75,0.985255,...,Sweep_10,M03_JS_A1_C01,A,3,PFC,6.3,F,9.96,Macaca fascicularis,M03
14,2.058032,0.60,-39.135743,11.889649,0.55,3.807068,6.171128,-52.532920,21.95,0.963031,...,Sweep_11,M03_JS_A1_C01,A,3,PFC,6.3,F,9.96,Macaca fascicularis,M03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63855,1.846199,1.35,-42.471191,45.846680,2.45,0.442505,0.359868,-78.247398,121.70,0.923707,...,Sweep_9,M21_SA_A1_C07,S,3,V1,8.7,M,7.65,Macaca fascicularis,M21
63856,2.099518,1.45,-39.785645,42.367676,2.55,0.381470,0.299768,-84.266822,0.00,0.944645,...,Sweep_9,M21_SA_A1_C07,S,3,V1,8.7,M,7.65,Macaca fascicularis,M21
63857,2.099518,1.45,-39.785645,42.367676,2.55,0.381470,0.299768,-84.266822,255.00,0.944645,...,Sweep_9,M21_SA_A1_C07,S,3,V1,8.7,M,7.65,Macaca fascicularis,M21
63858,2.276198,1.55,-37.100098,40.139893,2.65,0.335693,0.304954,-81.487302,359.55,0.930923,...,Sweep_9,M21_SA_A1_C07,S,3,V1,8.7,M,7.65,Macaca fascicularis,M21


### Create groups

In [27]:


# Identify metadata columns (excluding 'Monkey ID')
metadata_cols = ['dendriticType', 'SomaLayerLoc', 'brainOrigin', 'Sex', 'Species']


# Function to create subcomparisons for varying metadata
def create_comparison_groups(df, metadata_cols):
    results = []

    for varying_col in metadata_cols:
        # Columns to keep fixed (all except the varying one)
        fixed_cols = [col for col in metadata_cols if col != varying_col]

        # Group data by fixed metadata columns
        grouped = df.groupby(fixed_cols)
        subcomparison_index = 1

        for group_name, group_data in grouped:
            # Get unique values for the varying column within the fixed group
            varying_groups = group_data[varying_col].unique()

            # Ensure there are at least two values for the varying column
            if len(varying_groups) > 1:
                # Create pairwise subcomparisons
                for i in range(len(varying_groups)):
                    for j in range(i + 1, len(varying_groups)):
                        value_1 = varying_groups[i]
                        value_2 = varying_groups[j]

                        # Subset the data for the two groups being compared
                        group_1 = group_data[group_data[varying_col] == value_1]
                        group_2 = group_data[group_data[varying_col] == value_2]

                        # Append the results for this subcomparison
                        results.append({
                            'Subcomparison': subcomparison_index,
                            'Varying Metadata': varying_col,
                            'Fixed Metadata': ', '.join([f"{col}={val}" for col, val in zip(fixed_cols, group_name)]) if isinstance(group_name, tuple) else f"{fixed_cols[0]}={group_name}",
                            'Group 1 Value': value_1,
                            'Group 1 Count': len(group_1),
                            'Group 2 Value': value_2,
                            'Group 2 Count': len(group_2)
                        })
                        
                        subcomparison_index += 1

    return pd.DataFrame(results)

# Run the function to create comparison groups
grouping_results_df = create_comparison_groups(allMonkey_df, metadata_cols)

# Display the table in chunks if it is too large
if len(grouping_results_df) > 50:
    for i in range(0, len(grouping_results_df), 50):
        display(grouping_results_df.iloc[i:i+50])
else:
    display(grouping_results_df)


,Subcomparison,Varying Metadata,Fixed Metadata,Group 1 Value,Group 1 Count,Group 2 Value,Group 2 Count
0,1,dendriticType,"SomaLayerLoc=2, brainOrigin=PFC, Sex=F, Specie...",A,608,S,206
1,2,dendriticType,"SomaLayerLoc=2, brainOrigin=PFC, Sex=M, Specie...",S,4,A,252
2,3,dendriticType,"SomaLayerLoc=3, brainOrigin=PFC, Sex=F, Specie...",A,622,S,842
3,4,dendriticType,"SomaLayerLoc=3, brainOrigin=PFC, Sex=M, Specie...",A,1404,S,902
4,5,dendriticType,"SomaLayerLoc=3, brainOrigin=V1, Sex=M, Species...",S,1468,A,3032
5,6,dendriticType,"SomaLayerLoc=3, brainOrigin=V1, Sex=M, Species...",S,1468,NA,6
6,7,dendriticType,"SomaLayerLoc=3, brainOrigin=V1, Sex=M, Species...",A,3032,NA,6
7,8,dendriticType,"SomaLayerLoc=3, brainOrigin=V1, Sex=M, Species...",S,194,A,1286
8,9,dendriticType,"SomaLayerLoc=3, brainOrigin=V1, Sex=M, Species...",S,194,NA,40
9,10,dendriticType,"SomaLayerLoc=3, brainOrigin=V1, Sex=M, Species...",A,1286,NA,40


,Subcomparison,Varying Metadata,Fixed Metadata,Group 1 Value,Group 1 Count,Group 2 Value,Group 2 Count
50,35,SomaLayerLoc,"dendriticType=S, brainOrigin=V1, Sex=M, Specie...",3,1468,4,462
51,36,SomaLayerLoc,"dendriticType=S, brainOrigin=V1, Sex=M, Specie...",2,582,2_3,270
52,37,SomaLayerLoc,"dendriticType=S, brainOrigin=V1, Sex=M, Specie...",2,582,4,462
53,38,SomaLayerLoc,"dendriticType=S, brainOrigin=V1, Sex=M, Specie...",2_3,270,4,462
54,1,brainOrigin,"dendriticType=A, SomaLayerLoc=3, Sex=M, Specie...",PFC,1404,V1,3032
55,2,brainOrigin,"dendriticType=A, SomaLayerLoc=4, Sex=M, Specie...",PFC,120,V1,8
56,3,brainOrigin,"dendriticType=NA, SomaLayerLoc=NA, Sex=M, Spec...",PFC,5272,V1,584
57,4,brainOrigin,"dendriticType=NA, SomaLayerLoc=NA, Sex=M, Spec...",LIP,4,V1,334
58,5,brainOrigin,"dendriticType=S, SomaLayerLoc=2, Sex=M, Specie...",PFC,4,V1,582
59,6,brainOrigin,"dendriticType=S, SomaLayerLoc=3, Sex=M, Specie...",PFC,902,V1,1468
